In [ ]:
import json
from pathlib import Path

# Path to your JSON file
json_path = Path("data/thesis_data_1778418853489.json")

# Load the JSON
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("JSON loaded successfully!")
print(type(data))

In [ ]:
import pprint

pp = pprint.PrettyPrinter(depth=3)

pp.pprint(data[0])  # or data if dict

Ok so that worked. The problem now is, as we can see from the pretty print, that each sample has some nested stepData inside. This means we can't just put it in a dataframe and need to process it first.

In [ ]:
import pandas as pd

# Create a simplified version of the data
simplified_data = []

for item in data:
    simplified_item = item.copy()

    # Create a new column with the number of steps
    simplified_item["numSteps"] = len(item.get("stepData", []))

    # Remove the original huge stepData field
    del simplified_item["stepData"]

    simplified_data.append(simplified_item)

# Convert to DataFrame
df = pd.DataFrame(simplified_data)

# Preview
print(df.head())

Ok things are at least graspable now. Ofc this isn't helpful for the project, so lets see what we wanna do with the data.

Next we should crop parts of the steps that have constant size, because neural networks work better with that (or maybe can only work with that even idk)

In [ ]:
import random

WINDOW_SIZE = 60

samples = []

for item in data:

    step_data = item.get("stepData", [])

    # Skip sequences that are too short
    if len(step_data) < WINDOW_SIZE:
        continue

    # Pick random starting point
    max_start = len(step_data) - WINDOW_SIZE
    start_index = random.randint(0, max_start)

    # Extract fixed-size window
    window = step_data[start_index:start_index + WINDOW_SIZE]

    samples.append({
        "fastBuilderMode": item["fastBuilderMode"],
        "idAnonymous": item["idAnonymous"],
        "window": window
    })

print("Number of samples:", len(samples))

# TESTING ONLY
df2 = pd.DataFrame(samples)
df2["window"] = df2["window"].apply(len)
print(df2.head())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Only look at NORMAL mode replays for now
normal_samples = [s for s in samples if s["fastBuilderMode"] == "NORMAL"]

# Positions as an array of shape (samples, WINDOW_SIZE, 3) with x, y, z
positions = np.array([
    [[step["playerPosition"]["x"], step["playerPosition"]["y"], step["playerPosition"]["z"]]
     for step in s["window"]]
    for s in normal_samples
])

# Movement per step: sqrt(dx² + dy² + dz²) relative to the previous step
deltas = np.diff(positions, axis=1)
movement = np.sqrt((deltas ** 2).sum(axis=2))  # shape (samples, WINDOW_SIZE - 1)

print("NORMAL samples:", len(normal_samples))

steps = np.arange(1, WINDOW_SIZE)

plt.figure(figsize=(12, 5))
for m in movement:
    plt.plot(steps, m, color="gray", alpha=0.1, linewidth=0.8)
plt.plot(steps, movement.mean(axis=0), color="tab:blue", linewidth=2, label="mean")
plt.xlabel("step in window")
plt.ylabel("movement (blocks per step)")
plt.title(f"Movement per step, NORMAL mode ({len(normal_samples)} windows)")
plt.legend()
plt.show()

Now lets turn all windows into one "wide" DataFrame that a simple (non-sequential) neural network can take: one row per window, one column per feature per step.

- `movement_1 .. movement_59`: blocks moved since the previous step (like in the plot above)
- `yawChange_*` / `pitchChange_*`: how much the player turned their head since the previous step
- `clicks_0 .. clicks_59`, `sneaking_0 ..`, `blocksChanged_0 ..`: the raw per-step values

The change columns start at 1 because 60 steps only give 59 differences. Absolute positions are left out on purpose, because they mostly say where on the map someone is, not how they play.

In [ ]:
def per_step(get_value):
    """Array of shape (samples, WINDOW_SIZE): get_value(step) for every step of every window."""
    return np.array([[get_value(step) for step in s["window"]] for s in samples], dtype=float)

def step_columns(name, values, first_step):
    """Turn a (samples, steps) array into columns name_<first_step>, name_<first_step + 1>, ..."""
    columns = [f"{name}_{i}" for i in range(first_step, first_step + values.shape[1])]
    return pd.DataFrame(values, columns=columns)

x = per_step(lambda step: step["playerPosition"]["x"])
y = per_step(lambda step: step["playerPosition"]["y"])
z = per_step(lambda step: step["playerPosition"]["z"])
yaw = per_step(lambda step: step["playerPosition"]["yaw"])
pitch = per_step(lambda step: step["playerPosition"]["pitch"])

# Changes between consecutive steps, shape (samples, WINDOW_SIZE - 1)
movement = np.sqrt(np.diff(x, axis=1) ** 2 + np.diff(y, axis=1) ** 2 + np.diff(z, axis=1) ** 2)
# yaw is not limited to -180..180 in the data, so wrap the change to the shortest turn (-180..180)
yaw_change = (np.diff(yaw, axis=1) + 180) % 360 - 180
pitch_change = np.diff(pitch, axis=1)

clicks = per_step(lambda step: step["clicks"])
sneaking = per_step(lambda step: step["sneaking"])  # True/False -> 1.0/0.0
blocks_changed = per_step(lambda step: len(step["blocksChanged"]))

ping = per_step(lambda step: step["ping"])
ping_average = ping.mean(axis=1)
ping_jitter = np.abs(np.diff(ping, axis=1)).mean(axis=1)

wide_df = pd.concat([
    pd.DataFrame({
        "idAnonymous": [s["idAnonymous"] for s in samples], # remove that for clustering
        "fastBuilderMode": [s["fastBuilderMode"] for s in samples], # remove that for clustering
        "pingAvg": ping_average, # remove that for clustering
        "pingJitter": ping_jitter, # remove that for clustering
    }),
    step_columns("movement", movement, 1),
    step_columns("yawChange", yaw_change, 1),
    step_columns("pitchChange", pitch_change, 1),
    step_columns("clicks", clicks, 0),
    step_columns("sneaking", sneaking, 0),
    step_columns("blocksChanged", blocks_changed, 0),
], axis=1)

print(wide_df.shape)  # (samples, 4 + 3 * (WINDOW_SIZE - 1) + 3 * WINDOW_SIZE)

# Preview only: show the id/label columns and steps 0-2 of each feature (wide_df itself keeps all columns)
wide_df.filter(regex=r"^(idAnonymous|fastBuilderMode|pingAvg|pingJitter)$|_[0-2]$").head()